In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_parquet('../data/processed/cleaned.parquet')
print(data.shape)

(368014, 24)


In [3]:
print(data.columns)
data = data.sort_values("period").reset_index(drop=True)

Index(['period', 'Регистратор', 'contractor', 'КонтрагентИНН', 'account_dt',
       'СубконтоДт1', 'ВидСубконтоДт1', 'СубконтоДт2', 'ВидСубконтоДт2',
       'СубконтоДт3', 'ВидСубконтоДт3', 'account_kt', 'СубконтоКт1',
       'ВидСубконтоКт1', 'СубконтоКт2', 'ВидСубконтоКт2', 'СубконтоКт3',
       'ВидСубконтоКт3', 'dept_dt', 'dept_kt', 'amount', 'Содержание',
       'ТипДокумента', 'is_manual'],
      dtype='object')


# Создание фичей

## 1 временные

In [4]:
data['hour'] = data['period'].dt.hour
data['day_of_week'] = data['period'].dt.dayofweek
data['month'] = data['period'].dt.month
data['is_weekend'] = data['day_of_week'].isin([5, 6]).astype(int)
data['is_night'] = ((data['hour'] >= 20) | (data['hour'] < 8)).astype(int)
data['is_mounth_end'] = (data["period"].dt.day >= 25).astype(int)
data["is_quarter_end"] = (data["month"].isin([3, 6, 9, 12]) & (data["period"].dt.day >=
25)).astype(int)

## 2 сумовые признаки

In [5]:
data['log_amount'] = np.log1p(data['amount'].abs())
data["is_negative_amount"] = (data["amount"] < 0).astype(int)
data["abs_amount"] = data["amount"].abs()

## 3 признаки по паре счетов

In [6]:
data['account_pair'] = data['account_dt'].astype(str) + '_' + data['account_kt'].astype(str)

pair_freq = data['account_pair'].value_counts().to_dict()
data['pair_frequency'] = data['account_pair'].map(pair_freq)
data["log_pair_frequency"] = np.log1p(data["pair_frequency"])

data["is_rare_pair"] = (data["pair_frequency"] < 10).astype(int)


pair_stats = data.groupby("account_pair")["abs_amount"].agg(["mean", "std"]).reset_index()
pair_stats.columns = ["account_pair", "pair_mean", "pair_std"]
data = data.merge(pair_stats, on="account_pair", how="left")


data["amount_zscore"] = np.where(data["pair_std"] > 0, (data["abs_amount"] - data["pair_mean"]) / data["pair_std"], 0)

# Сначала обнуляем z-score для редких пар, потом считаем is_amount_outlier
data.loc[data["pair_frequency"] < 5, "amount_zscore"] = 0
data["is_amount_outlier"] = (data["amount_zscore"].abs() > 3).astype(int)


## 4 контрагенты

In [7]:
contractor_frequency = data['contractor'].value_counts().to_dict()
data['contractor_freq'] = data['contractor'].map(contractor_frequency)
data["log_contractor_frequency"] = np.log1p(data["contractor_freq"])

data["contractor_ops_before"] = data.groupby("contractor").cumcount()
data["is_first_operation"] = (data["contractor_ops_before"] == 0).astype(int)

## 5 по подразделениям

In [8]:
freq_dt = data['dept_dt'].value_counts(normalize=True).to_dict()
data['dept_dt_freq'] = data['dept_dt'].map(freq_dt)

freq_kt = data['dept_kt'].value_counts(normalize=True).to_dict()
data['dept_kt_freq'] = data['dept_kt'].map(freq_kt)

## 6 комбинированные

Крупная сумма + редкая пара счетов

Крупная сумма + нерабочее время

Новый контрагент + крупная сумма

Ручная проводка + крупная сумма (если есть признак is_manual)

Первая операция с контрагентом по данной паре счетов

In [9]:
data.columns

Index(['period', 'Регистратор', 'contractor', 'КонтрагентИНН', 'account_dt',
       'СубконтоДт1', 'ВидСубконтоДт1', 'СубконтоДт2', 'ВидСубконтоДт2',
       'СубконтоДт3', 'ВидСубконтоДт3', 'account_kt', 'СубконтоКт1',
       'ВидСубконтоКт1', 'СубконтоКт2', 'ВидСубконтоКт2', 'СубконтоКт3',
       'ВидСубконтоКт3', 'dept_dt', 'dept_kt', 'amount', 'Содержание',
       'ТипДокумента', 'is_manual', 'hour', 'day_of_week', 'month',
       'is_weekend', 'is_night', 'is_mounth_end', 'is_quarter_end',
       'log_amount', 'is_negative_amount', 'abs_amount', 'account_pair',
       'pair_frequency', 'log_pair_frequency', 'is_rare_pair', 'pair_mean',
       'pair_std', 'amount_zscore', 'is_amount_outlier', 'contractor_freq',
       'log_contractor_frequency', 'contractor_ops_before',
       'is_first_operation', 'dept_dt_freq', 'dept_kt_freq'],
      dtype='object')

In [10]:
data['large_and_rare'] = ((data['amount_zscore'].abs() > 2) & (data['is_rare_pair'] == 1)).astype(int)

data['late_and_large'] = ((data['is_night'] == 1) & (data['amount_zscore'].abs() > 2)).astype(int)

data['new_cont_and_large'] = ((data['is_first_operation'] == 1) & (data['amount_zscore'].abs() > 2)).astype(int)

data['manual_and_large'] = ((data['is_manual'] == 1) & (data['amount_zscore'].abs() > 2)).astype(int)

data["first_contractor_pair"] = data.groupby(["contractor", "account_pair"]).cumcount()
data["is_first_contractor_pair"] = (data["first_contractor_pair"] == 0).astype(int)

## 7 системные признаки

In [11]:
data["date"] = data["period"].dt.date
# Количество проводок в тот же день
daily_count = data.groupby("date").size().to_dict()
data["daily_volume"] = data["date"].map(daily_count)
# Количество проводок в тот же час
data["hour_key"] = data["period"].dt.strftime("%Y-%m-%d %H")
hourly_count = data.groupby("hour_key").size().to_dict()
data["hourly_volume"] = data["hour_key"].map(hourly_count)

## 8 кодирование категорий

In [14]:
for col in ["account_dt", "account_kt", "account_pair", "ТипДокумента"]:
    freq = data[col].value_counts(normalize=True).to_dict()
    data[col + "_freq"] = data[col].map(freq)

## 9 матрица фичей

In [17]:
feature_cols = [
    # Временные
    "hour", "day_of_week", "month", "is_weekend", "is_night",
    "is_mounth_end", "is_quarter_end",
    # Суммовые
    "log_amount", "is_negative_amount", "amount_zscore", "is_amount_outlier",
    # Пары счетов
    "log_pair_frequency", "is_rare_pair",
    "account_dt_freq", "account_kt_freq", "account_pair_freq",
    # Контрагенты
    "log_contractor_frequency", "is_first_operation",
    "is_first_contractor_pair",
    # Комбинированные
    "large_and_rare", "late_and_large", "new_cont_and_large", "manual_and_large",
    # Системные
    "daily_volume", "hourly_volume",
    # Тип документа
    "ТипДокумента_freq",

    "is_manual",
]

X = data[feature_cols].copy()
X = X.fillna(0)
print(f"Матрица фичей: {X.shape}")
print(f"NaN в матрице: {X.isnull().sum().sum()}")


X.to_parquet("../data/features/feature_matrix.parquet", index=False)
data.to_parquet("../data/features/data_with_features.parquet", index=False)

Матрица фичей: (368014, 27)
NaN в матрице: 0
